# Neural Network from Scratch
## Neural networks for energy applications — Day 1, Exercise 2

In this exercise we implement a **2-layer neural network** in pure NumPy to classify sonar signals as Rock or Mine. No deep-learning frameworks are used — we code every component by hand so you understand exactly what happens under the hood.

The dataset (`sonar.all-data.csv`) contains 208 samples with 60 continuous features.

By the end of this notebook you will:
- Implement the sigmoid activation and its derivative
- Initialize weights with proper scaling
- Code forward propagation through two layers
- Compute binary cross-entropy loss
- Implement backpropagation to compute gradients, and check them numerically
- Train the network with gradient descent and evaluate it **once** on the test set
- Explore hyperparameters on a validation split and compare with sklearn's `MLPClassifier`

---
### 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn is used only for splitting, scaling, metrics and the final comparison
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier

np.random.seed(42)
print("Imports complete.")

---
### 2 — Load and Prepare the Sonar Dataset

Same dataset as Lab 1: 60 features, binary label (R/M). We use the same explicit label mapping (**Mine = 1**, the positive class, **Rock = 0**) and the identical 80/20 split call, so the 42 test echoes are the same as in Lab 1 and the scores can be compared on equal terms. Features are standardized with a scaler fitted on the training rows only; we keep the unscaled training rows for the validation experiments in section 12.

**Tasks:**
- Load `sonar.all-data.csv` (no header)
- Encode labels with the explicit mapping `LABEL_MAP = {"R": 0, "M": 1}` (Mine = 1, as in Lab 1) and print it
- Split the row numbers 80/20 exactly as in Lab 1: `train_test_split(np.arange(len(y)), test_size=0.2, random_state=42, stratify=y_raw)`
- Keep the unscaled rows as `X_train_raw`, `X_test_raw`; standardize features (fit on train only) into `X_train`, `X_test`
- Reshape y to column vectors `(-1, 1)`

In [ ]:
# YOUR CODE HERE
# Hint: LABEL_MAP = {"R": 0, "M": 1}; y = df["label"].map(LABEL_MAP).to_numpy()

In [ ]:
# YOUR CODE HERE
# Split (same call as Lab 1), scale, reshape y
# Hint: idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42, stratify=y_raw)
# Hint: y_train = y_train.reshape(-1, 1)

---
### 3 — Sigmoid Activation Function

The sigmoid function maps any real number to the range $(0, 1)$:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Its derivative (needed for backpropagation) is:

$$\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$$

**Tasks:**
- Implement `sigmoid(z)` — use `np.clip` to avoid overflow
- Implement `sigmoid_derivative(a)` where `a = sigmoid(z)`
- Verify `sigmoid(0) = 0.5`
- Plot both functions over the range [-6, 6]

In [ ]:
def sigmoid(z):
    """Compute the sigmoid activation."""
    # YOUR CODE HERE
    pass


def sigmoid_derivative(a):
    """Compute sigmoid derivative given the activation output a = sigmoid(z)."""
    # YOUR CODE HERE
    pass


# Test and plot
# YOUR CODE HERE


---
### 4 — Weight Initialization

We initialize weights from a normal distribution scaled by $\frac{1}{\sqrt{n_{\text{in}}}}$ to prevent the activations from exploding or vanishing at the start of training. Biases are initialized to zero.

**Tasks:**
- Implement `initialize_weights(n_input, n_hidden, n_output)` returning a dict with keys `'W1'`, `'b1'`, `'W2'`, `'b2'`
- W1 shape: `(n_input, n_hidden)`, scaled by `1/sqrt(n_input)`
- W2 shape: `(n_hidden, n_output)`, scaled by `1/sqrt(n_hidden)`
- Biases: zeros with appropriate shapes

In [ ]:
def initialize_weights(n_input, n_hidden, n_output):
    """Initialize weights and biases for a 2-layer network.

    Returns
    -------
    dict with keys 'W1', 'b1', 'W2', 'b2'
    """
    # YOUR CODE HERE
    pass


# Test
# YOUR CODE HERE


---
### 5 — Forward Propagation

Our 2-layer network computes:

**Hidden layer:**
$$z_1 = X \cdot W_1 + b_1, \quad a_1 = \sigma(z_1)$$

**Output layer:**
$$z_2 = a_1 \cdot W_2 + b_2, \quad a_2 = \sigma(z_2)$$

where $a_2$ is the predicted probability.

**Tasks:**
- Implement `forward(X, params)` returning a cache dict with `'z1'`, `'a1'`, `'z2'`, `'a2'`
- Use `@` for matrix multiplication

In [ ]:
def forward(X, params):
    """Perform forward propagation.

    Returns
    -------
    cache : dict with 'z1', 'a1', 'z2', 'a2'
    """
    # YOUR CODE HERE
    pass


# Test
# YOUR CODE HERE


---
### 6 — Binary Cross-Entropy Loss

For binary classification the loss function is:

$$\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m}\left[y_i\log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$$

**Tasks:**
- Implement `binary_cross_entropy(y_true, y_pred)`
- Clip predictions with `np.clip(y_pred, 1e-15, 1-1e-15)` to avoid `log(0)`
- Test: a confident correct prediction should give a low loss, a confident wrong one a high loss

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    """Compute binary cross-entropy loss."""
    # YOUR CODE HERE
    pass


# Test
# YOUR CODE HERE


---
### 7 — Backward Propagation

We compute the gradients of the loss with respect to each parameter using the chain rule.

**Output layer gradients:**
$$dz_2 = a_2 - y$$
$$dW_2 = \frac{1}{m} a_1^T \cdot dz_2$$
$$db_2 = \frac{1}{m} \sum dz_2$$

**Hidden layer gradients:**
$$dz_1 = (dz_2 \cdot W_2^T) \odot \sigma'(z_1), \qquad \sigma'(z_1) = a_1 \odot (1 - a_1)$$
$$dW_1 = \frac{1}{m} X^T \cdot dz_1$$
$$db_1 = \frac{1}{m} \sum dz_1$$

`sigmoid_derivative(a1)` computes $\sigma'(z_1)$ from the stored activation $a_1$.

Two checks follow: every gradient must have the shape of its parameter (asserted below), and a **numerical gradient check** (the Part 6 slide) compares the backpropagated gradient with the slope of the loss when one weight is nudged by ±ε. Use float64 arrays (ours are) and restore each perturbed weight. Relative errors around 1e-7 or smaller mean the backward pass is right; errors near 1 mean a bug.

**Tasks:**
- Implement `backward(X, y, params, cache)` returning a grads dict with `'dW1'`, `'db1'`, `'dW2'`, `'db2'`
- Assert that every gradient has the shape of its parameter
- In the next cell, add the gradient check from the Part 6 slide and report the three relative errors

In [ ]:
def backward(X, y, params, cache):
    """Perform backward propagation and return gradients.

    Returns
    -------
    grads : dict with 'dW1', 'db1', 'dW2', 'db2'
    """
    # YOUR CODE HERE
    pass


# Test: assert that every gradient has the shape of its parameter
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE
# Numerical gradient check (Part 6 slide): write grad_check(loss_fn, params, grads, key, idx, eps=1e-5)
# and check W1 (0, 0), b1 (0, 3) and W2 (5, 0) at initialization (np.random.seed(42), 32 hidden units).
# Hint: loss_fn = lambda p: binary_cross_entropy(y_train, forward(X_train, p)["a2"])
#       grads = backward(X_train, y_train, params, forward(X_train, params))

---
### 8 — Training Loop

We tie everything together: initialize weights, then for each epoch perform forward propagation, compute the loss, run backpropagation, and update the weights.

**Tasks:**
- Implement `train(X, y, n_hidden, learning_rate, epochs, verbose=True)`
- Inside the loop: forward -> loss -> backward -> update weights
- Update rule: `W -= learning_rate * dW` (same for biases)
- Return the trained params dict and a list of loss values

In [ ]:
def train(X, y, n_hidden, learning_rate, epochs, verbose=True):
    """Train a 2-layer neural network.

    Returns
    -------
    params       : dict, trained weights
    loss_history : list of float
    """
    # YOUR CODE HERE
    pass


---
### 9 — Train the Network

Train with 1000 epochs, a learning rate of 0.01, and 32 hidden units.

**Tasks:**
- Set `np.random.seed(42)` for reproducibility
- Call `train()` with the specified hyperparameters

In [ ]:
# YOUR CODE HERE


---
### 10 — Plot Training Loss Curve

**Tasks:**
- Plot `loss_history` vs epoch
- Print the initial and final loss values

In [ ]:
# YOUR CODE HERE


---
### 11 — Evaluate on the Test Set

The settings in section 9 (32 hidden units, learning rate 0.01, 1000 epochs, seed 42) were fixed before looking at any score, so this is a legitimate **one-time** test of a locked choice: the only use of the test set in this notebook. The 42 test echoes are the same as in Lab 1, so compare with the Lab 1 result on equal terms. Each echo is worth 2.4 percentage points.

**Tasks:**
- Implement `predict(X, params, threshold=0.5)` that runs forward propagation and thresholds the output
- Compute and print training and test accuracy, with the number of correct rows
- Print the classification report for the test set

In [ ]:
def predict(X, params, threshold=0.5):
    """Predict binary labels using the trained network."""
    # YOUR CODE HERE
    pass


# YOUR CODE HERE
# Compute train/test accuracy (with counts) and print the classification report

---
### 12 — Experiment with Hyperparameters (on a validation split)

Let us see how different choices of hidden layer size, learning rate, and number of epochs affect performance. **These comparisons use a validation split carved from the training rows, not the test set**: choosing settings by test accuracy would turn the test set into a second validation set, and the "best" score would be optimistic.

We hold out 25% of the 166 training rows (42 echoes) as validation data, refit the scaler on the remaining 124 rows, train each setting on those 124 rows, and score it on the validation rows. Validation accuracy on 42 echoes moves in steps of 2.4 points, so we also print the validation loss.

The test score in section 11 belongs to the settings fixed in advance. If a different setting looks better here, report its validation score; do not re-test it on the same 42 test echoes.

**Tasks:**
- Run the next cell (provided): it carves the validation split and defines `run_setting(n_hidden, learning_rate, epochs)`
- Experiment 1: Vary `hidden_size` in `[8, 16, 32, 64, 128]` (lr=0.01, epochs=1000)
- Experiment 2: Vary `learning_rate` in `[0.001, 0.005, 0.01, 0.05, 0.1]` (hidden=32, epochs=1000)
- Experiment 3: Vary `epochs` in `[100, 500, 1000, 2000, 5000]` (hidden=32, lr=0.01)
- Print the training loss, validation loss, and validation accuracy for each setting
- Plot the validation accuracy of all three experiments as line charts in a 1x3 subplot

In [ ]:
# Validation split carved from the TRAINING rows only; the test rows stay untouched
X_sub_raw, X_val_raw, y_sub, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.25, random_state=42, stratify=y_train.ravel()
)
val_scaler = StandardScaler().fit(X_sub_raw)        # fitted on the 124 sub-training rows only
X_sub = val_scaler.transform(X_sub_raw)
X_val = val_scaler.transform(X_val_raw)
print(f"Sub-training rows: {X_sub.shape[0]}, validation rows: {X_val.shape[0]}")


def run_setting(n_hidden, learning_rate, epochs):
    """Train on the sub-training rows (seed 42); return final training loss, validation loss and accuracy."""
    np.random.seed(42)
    p, losses = train(X_sub, y_sub, n_hidden=n_hidden, learning_rate=learning_rate,
                      epochs=epochs, verbose=False)
    val_loss = binary_cross_entropy(y_val, forward(X_val, p)["a2"])
    val_acc = accuracy_score(y_val, predict(X_val, p))
    return losses[-1], val_loss, val_acc

In [ ]:
# YOUR CODE HERE
# Experiment 1: Vary hidden size
# Hint: train_loss, val_loss, val_acc = run_setting(h, 0.01, 1000)

In [ ]:
# YOUR CODE HERE
# Experiment 2: Vary learning rate

In [ ]:
# YOUR CODE HERE
# Experiment 3: Vary epochs

In [ ]:
# YOUR CODE HERE
# Plot the validation accuracy of all three experiments in a 1x3 subplot

**Your interpretation:** Which setting would you choose, and how confident are you, given 42 validation echoes? If you adopted it, which score would you report for it, and why not a new score on the test rows?

---
### 13 — Compare with sklearn MLPClassifier

As a sanity check, we compare our from-scratch network against sklearn's `MLPClassifier` with a similar architecture. Both are trained on the same 124 sub-training rows and scored on the same validation rows as section 12 (run its first cell before this section). The test set is not used again.

**Tasks:**
- Train our network on `X_sub`, `y_sub` with the section 9 settings (seed 42)
- Create an `MLPClassifier` with `hidden_layer_sizes=(32,)`, `activation="logistic"`, `solver="sgd"`, `learning_rate_init=0.01`, `max_iter=1000`, `random_state=42` and fit it on the same rows
- Print a comparison table of training and validation accuracy (scratch vs. sklearn)
- Create a grouped bar chart comparing training and validation accuracy for both

In [ ]:
# YOUR CODE HERE


---
### Summary

In this exercise we:

1. Loaded the UCI Sonar dataset with the explicit mapping Mine = 1, Rock = 0, used the same split as Lab 1, and standardized it with a scaler fitted on the training rows.
2. Implemented the **sigmoid** activation and its derivative.
3. Wrote a weight initialization function scaled by $1/\sqrt{n}$.
4. Coded **forward propagation** for a 2-layer network.
5. Implemented **binary cross-entropy** loss.
6. Derived and coded **backpropagation** to compute all four gradients ($dW_1, db_1, dW_2, db_2$), asserted their shapes, and checked them numerically.
7. Built a training loop with gradient descent.
8. Trained the network and plotted the loss curve.
9. Evaluated once on the test set (the same 42 echoes as Lab 1).
10. Explored the effect of hidden size, learning rate, and epochs on a validation split carved from the training rows.
11. Compared our implementation against sklearn's `MLPClassifier` on the same validation rows.

**Key takeaway:** A neural network is fundamentally a sequence of linear transformations and non-linear activations. Backpropagation is just the chain rule applied systematically. Understanding these mechanics prepares you to work with deep-learning frameworks like PyTorch and TensorFlow, where the same operations happen — just with automatic differentiation.